<p style="text-align:right"><img src="https://www.iesb.br/content/themes/iesb-chleba-themosis/assets/imagens/logo.png" width="100"/></p>

# 🧠 Laboratório 8 — Prompt e saída estruturada

**Inteligência Artificial (MDC050)** · Centro Universitário IESB · Turma ENGCNM2B
**Prof. Rodrigo Gonçalves Pinto** · Aula 09 · 21/10/2026

---

## O que você vai fazer aqui

Hoje o Qwen vai aprender a transformar um pedido em português nos **dados de entrada do A\***. No fim, você terá a primeira versão do agente: o usuário escreve, o modelo extrai a origem e o destino, o código confere e o A\* calcula.

| Etapa | O que acontece |
|---|---|
| 0 | Preparar o Colab e carregar o Qwen |
| 1 | Um prompt sem formato: a resposta não serve para o Python |
| 2 | Ler JSON em Python e conferir os dados |
| 3 | Um prompt com formato JSON |
| 4 | Ensinar com exemplos (few-shot) |
| 5 | Versionar os prompts e medir a taxa de acerto |
| 6 | Ligar o modelo ao A\* |
| Desafio | Tudo de novo, no domínio do seu trio |

## Como entregar

1. Complete os **seis TODOs**. Embaixo de cada um há uma célula de verificação que imprime ✅ ou ❌.
2. Responda a célula **Sua análise**, no final.
3. **Antes de entregar:** `Ambiente de execução → Reiniciar sessão e executar tudo`.
4. Envie o link do notebook pelo Google Sala de Aula.

> **Crie sua cópia antes de começar:** `Arquivo → Salvar uma cópia no Drive`.

---
# Etapa 0 — Preparação

## Ligar a GPU

`Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU → Salvar`.
Se não houver GPU disponível, escolha **CPU**: tudo funciona, só fica mais lento.

In [ ]:
import torch
import json

if torch.cuda.is_available():
    dispositivo = "cuda"
    print("✅ GPU ligada.")
else:
    dispositivo = "cpu"
    print("⚠️ Sem GPU. Funciona, mas as respostas do Qwen vão demorar mais.")

## Token do Hugging Face

Você criou o token no Laboratório 7. Se ainda não criou, o resumo é:

1. **huggingface.co** → foto (canto superior direito) → **Settings** → **Access Tokens** → **Create new token**.
2. **Token type:** `Read`. Dê um nome (ex.: `colab-ia`) → **Create token** → copie o token (começa com `hf_`).
3. No Colab, ícone da **chave 🔑 (Secrets)** → **Adicionar novo secret** → Nome `HF_TOKEN`, Valor: o token → ative **Acesso ao notebook**.

> O token fica nos Segredos, **nunca** escrito no código. Se não conseguir criar agora, pule a próxima célula: o Qwen baixa mesmo sem login.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
print("✅ Login no Hugging Face feito.")

## Carregar o Qwen e a função `conversar`

São as mesmas células do Laboratório 7. A carga demora cerca de 1 minuto.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

nome_qwen = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizador_qwen = AutoTokenizer.from_pretrained(nome_qwen)

if dispositivo == "cuda":
    qwen = AutoModelForCausalLM.from_pretrained(nome_qwen, dtype=torch.float16).to(dispositivo)
else:
    qwen = AutoModelForCausalLM.from_pretrained(nome_qwen, dtype=torch.float32)
qwen.eval()


def conversar(mensagens, max_tokens=60):
    entrada = tokenizador_qwen.apply_chat_template(
        mensagens, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(dispositivo)
    with torch.no_grad():
        saida = qwen.generate(**entrada, max_new_tokens=max_tokens, do_sample=False)
    tamanho_da_pergunta = entrada["input_ids"].shape[-1]
    return tokenizador_qwen.decode(saida[0][tamanho_da_pergunta:], skip_special_tokens=True)


print("✅ Qwen carregado em", dispositivo)

> **Por que `do_sample=False`?** Assim o modelo sempre escolhe o token mais provável: é a **temperatura zero** da Aula 08. Para extrair dados, queremos respostas repetíveis, não criativas.

---
# Etapa 1 — Um prompt sem formato

## O mapa e as cidades válidas

Usamos o mapa da Romênia das aulas de busca. O agente só aceita estas cidades:

In [ ]:
CIDADES = ["Arad", "Zerind", "Oradea", "Sibiu", "Timisoara", "Lugoj", "Mehadia",
           "Drobeta", "Craiova", "Rimnicu Vilcea", "Fagaras", "Pitesti", "Bucareste",
           "Giurgiu", "Urziceni", "Hirsova", "Eforie", "Vaslui", "Iasi", "Neamt"]

print(len(CIDADES), "cidades válidas")

## A função `montar_mensagens`

Ela junta as peças do prompt na lista de mensagens que o modelo recebe. Já está pronta:

- `regras` vai no papel **system**;
- `exemplos` (opcional) é uma lista de pares (pedido, resposta) que viram mensagens **user** e **assistant**;
- `pedido` vai no último **user**.

In [ ]:
def montar_mensagens(regras, pedido, exemplos=[]):
    mensagens = [{"role": "system", "content": regras}]
    for exemplo_pedido, exemplo_resposta in exemplos:
        mensagens.append({"role": "user", "content": exemplo_pedido})
        mensagens.append({"role": "assistant", "content": exemplo_resposta})
    mensagens.append({"role": "user", "content": pedido})
    return mensagens


# veja como fica (sem chamar o modelo)
for m in montar_mensagens("REGRAS AQUI", "PEDIDO AQUI", [("EXEMPLO DE PEDIDO", "EXEMPLO DE RESPOSTA")]):
    print(m)

## Prompt v1: só a pergunta

Vamos pedir a origem e o destino **sem dizer o formato** da resposta.

In [ ]:
PROMPT_V1 = "Diga de onde o usuário quer sair e para onde ele quer ir."

pedido = "preciso chegar em sibiu, estou em oradea"
resposta = conversar(montar_mensagens(PROMPT_V1, pedido))
print(resposta)

**Pergunta:** a resposta está correta para um humano. Mas como o Python vai descobrir, dentro desse texto, qual palavra é a origem e qual é o destino? Para o A\*, precisamos de **dados**, não de uma frase.

---
# Etapa 2 — JSON: dados em forma de texto

**JSON** é um formato de texto que representa dados. Ele tem a mesma cara de um dicionário Python:

| JSON | Python |
|---|---|
| `{"origem": "Arad"}` | `{"origem": "Arad"}` |
| `null` | `None` |
| `true` / `false` | `True` / `False` |
| só aspas **duplas** | aspas simples ou duplas |

O comando `json.loads(texto)` transforma o texto JSON em dicionário Python.

In [ ]:
texto = '{"origem": "Arad", "destino": "Bucareste"}'
dados = json.loads(texto)

print(type(texto), "->", type(dados))
print("Origem :", dados["origem"])
print("Destino:", dados["destino"])

## 🛠️ TODO 1 — Recortar o JSON da resposta

Às vezes o modelo escreve algo antes ou depois do JSON, por exemplo:

```
Claro! Aqui está: {"origem": "Arad", "destino": "Bucareste"} Boa viagem!
```

A função abaixo encontra a posição da primeira chave `{` e da última `}`. **Falta uma linha:** recortar o texto entre elas.

*Dica:* `trecho = texto[inicio:fim + 1]` (o `+ 1` inclui a própria `}`).

In [ ]:
def extrair_json(texto):
    inicio = texto.find("{")       # posição da primeira {   (-1 se não achar)
    fim = texto.rfind("}")         # posição da última }     (-1 se não achar)
    if inicio == -1 or fim == -1:
        return None                # não há JSON na resposta

    # TODO 1: recorte o texto da posição inicio até a posição fim (incluindo a })
    trecho = ""   # apague o "" e escreva o recorte

    try:
        return json.loads(trecho)  # tenta transformar em dicionário
    except:
        return None                # o trecho não era um JSON válido


print(extrair_json('Claro! Aqui está: {"origem": "Arad", "destino": "Bucareste"} Boa viagem!'))

In [ ]:
# --- Verificação do TODO 1 (não altere) ---
r1 = extrair_json('Claro! {"origem": "Arad", "destino": "Bucareste"} Boa viagem!')
r2 = extrair_json('```json\n{"origem": "Oradea", "destino": null}\n```')
r3 = extrair_json("Não entendi o pedido.")
if r1 == {"origem": "Arad", "destino": "Bucareste"} and r2 == {"origem": "Oradea", "destino": None} and r3 is None:
    print("✅ TODO 1 correto. O JSON é recortado mesmo com texto em volta.")
else:
    print("❌ Ainda não. Confira o recorte texto[inicio:fim + 1].")

## 🛠️ TODO 2 — Conferir os dados antes de chamar o A\*

O modelo pode errar: devolver uma cidade que não existe no mapa, esquecer um campo ou mandar `null`. **Nunca** mandamos para o A\* sem conferir.

A função `validar` já confere se os dados existem e se têm os dois campos. **Falta a última conferência:** a origem **e** o destino precisam estar na lista de cidades válidas.

*Dica:* `return dados["origem"] in validas and dados["destino"] in validas`

In [ ]:
def validar(dados, validas):
    if dados is None:
        return False                              # não veio JSON
    if "origem" not in dados or "destino" not in dados:
        return False                              # faltou algum campo
    # TODO 2: devolva True só se a origem E o destino estiverem na lista 'validas'
    return None   # apague esta linha e escreva a sua


print(validar({"origem": "Arad", "destino": "Bucareste"}, CIDADES))
print(validar({"origem": "Arad", "destino": "Paris"}, CIDADES))

In [ ]:
# --- Verificação do TODO 2 (não altere) ---
casos = [({"origem": "Arad", "destino": "Bucareste"}, True),
         ({"origem": "Arad", "destino": "Paris"}, False),
         ({"origem": "Arad", "destino": None}, False),
         ({"origem": "arad", "destino": "Bucareste"}, False),
         ({"origem": "Arad"}, False),
         (None, False)]
erros = 0
for dados, esperado in casos:
    if validar(dados, CIDADES) != esperado:
        erros = erros + 1
        print("   errou em", dados, "- esperado", esperado)
if erros == 0:
    print("✅ TODO 2 correto. Repare: 'arad' em minúsculas é recusado — o nome tem que ser igual ao da lista.")
else:
    print("❌ Ainda não.", erros, "caso(s) errado(s).")

---
# Etapa 3 — Prompt v2: pedindo o formato JSON

Um bom prompt de extração tem quatro peças:

1. **Papel:** quem o modelo é.
2. **Tarefa:** o que ele deve fazer.
3. **Contexto:** a lista de cidades válidas (o modelo só sabe o que está no prompt).
4. **Formato:** exatamente como a resposta deve vir, e o que fazer quando não souber.

In [ ]:
PROMPT_V2 = (
    "Você extrai origem e destino de pedidos de rota. "                         # papel e tarefa
    "Cidades válidas: " + ", ".join(CIDADES) + ". "                             # contexto
    'Responda APENAS com JSON no formato {"origem": "...", "destino": "..."}. '  # formato
    "Escreva o nome da cidade exatamente como na lista. "
    "Se alguma cidade não estiver na lista, use null."
)

pedidos = [
    "Quero ir de Arad até Bucareste pelo caminho mais curto.",
    "preciso chegar em sibiu, estou em oradea",
    "Me leva de Arad para Paris.",
]

for pedido in pedidos:
    resposta = conversar(montar_mensagens(PROMPT_V2, pedido))
    dados = extrair_json(resposta)
    print(pedido)
    print("   resposta:", resposta)
    print("   válido para o A*?", validar(dados, CIDADES))
    print()

---
# Etapa 4 — Ensinar com exemplos (few-shot)

**Zero-shot** é pedir sem mostrar exemplos. **Few-shot** é mostrar alguns pares (pedido → resposta certa) antes do pedido real. O modelo imita o padrão dos exemplos.

Exemplos ajudam nos casos difíceis: nome em outra língua (*Bucharest*), ordem invertida, pedido sem destino.

## 🛠️ TODO 3 — Escreva mais um exemplo

A lista `EXEMPLOS` já tem dois pares. Acrescente **um terceiro** em que o usuário escreve o nome de uma cidade **em inglês** ou **sem maiúscula**, e a resposta traz o nome **como está na lista**.

Por exemplo: `("I am in Arad and want to go to Bucharest", '{"origem": "Arad", "destino": "Bucareste"}')`

In [ ]:
EXEMPLOS = [
    ("estou em pitesti e quero chegar em craiova", '{"origem": "Pitesti", "destino": "Craiova"}'),
    ("Quero ir para Lisboa saindo de Sibiu", '{"origem": "Sibiu", "destino": null}'),
    # TODO 3: acrescente aqui o terceiro exemplo (pedido, resposta)
]

print(len(EXEMPLOS), "exemplos")

In [ ]:
# --- Verificação do TODO 3 (não altere) ---
ok = len(EXEMPLOS) >= 3
for exemplo_pedido, exemplo_resposta in EXEMPLOS[2:]:
    dados = extrair_json(exemplo_resposta)
    if not validar(dados, CIDADES):
        ok = False
        print("   a resposta do exemplo não é um JSON com cidades válidas:", exemplo_resposta)
if ok:
    print("✅ TODO 3 correto.", len(EXEMPLOS), "exemplos, e as respostas são JSON com cidades da lista.")
else:
    print("❌ Ainda não. Acrescente um terceiro exemplo cuja resposta seja um JSON com duas cidades da lista.")

In [ ]:
PROMPT_V3 = PROMPT_V2      # as mesmas regras; a diferença são os EXEMPLOS

pedido = "Saindo de Timisoara, how do I get to Bucharest?"
print("Sem exemplos :", conversar(montar_mensagens(PROMPT_V2, pedido)))
print("Com exemplos :", conversar(montar_mensagens(PROMPT_V3, pedido, EXEMPLOS)))

---
# Etapa 5 — Versionar os prompts e medir

O Marco 2 exige **prompts versionados**: cada versão fica guardada com um nome, e você mede quantos casos de teste ela acerta. Assim você sabe se uma mudança melhorou ou piorou o agente.

Cada **caso de teste** é um pedido com a resposta esperada.

## 🛠️ TODO 4 — Acrescente dois casos de teste

A lista `CASOS` tem oito casos. Acrescente **mais dois**, escritos por você. Pelo menos um deles deve ser **difícil** (gíria, erro de digitação, ordem invertida, cidade fora do mapa…).

Formato: `(pedido, origem_esperada, destino_esperado)`. Use `None` quando o certo for `null`.

In [ ]:
CASOS = [
    ("Quero ir de Arad até Bucareste.", "Arad", "Bucareste"),
    ("Saindo de Timisoara, como chego em Pitesti?", "Timisoara", "Pitesti"),
    ("preciso chegar em sibiu, estou em oradea", "Oradea", "Sibiu"),
    ("Me leva de Arad para Paris.", "Arad", None),
    ("rota de craiova pra fagaras", "Craiova", "Fagaras"),
    ("Estou em Iasi. Destino: Eforie.", "Iasi", "Eforie"),
    ("How do I get from Zerind to Bucharest?", "Zerind", "Bucareste"),
    ("Quero ir até Giurgiu, partindo de Rimnicu Vilcea", "Rimnicu Vilcea", "Giurgiu"),
    # TODO 4: acrescente aqui mais dois casos (pedido, origem, destino)
]

print(len(CASOS), "casos de teste")

In [ ]:
# --- Verificação do TODO 4 (não altere) ---
if len(CASOS) >= 10:
    print("✅ TODO 4 correto.", len(CASOS), "casos de teste.")
else:
    print("❌ Ainda não. Acrescente mais", 10 - len(CASOS), "caso(s).")

## A função `avaliar`

Ela roda todos os casos com um prompt e conta os acertos. Um caso só conta como acerto se a origem **e** o destino vierem exatamente iguais aos esperados. Já está pronta.

> Na CPU, cada versão leva alguns minutos (são 10 chamadas ao modelo). Na GPU, alguns segundos.

In [ ]:
def avaliar(regras, exemplos, casos):
    acertos = 0
    for pedido, origem_esperada, destino_esperado in casos:
        dados = extrair_json(conversar(montar_mensagens(regras, pedido, exemplos)))
        if dados is not None and dados.get("origem") == origem_esperada and dados.get("destino") == destino_esperado:
            acertos = acertos + 1
            marca = "certo"
        else:
            marca = "ERRADO"
        print("  ", marca, "|", pedido, "->", dados)
    print("   Acertos:", acertos, "de", len(casos))
    return acertos


PROMPTS = {
    "v1": (PROMPT_V1, []),          # sem formato
    "v2": (PROMPT_V2, []),          # com formato JSON
    "v3": (PROMPT_V3, EXEMPLOS),    # com formato JSON e exemplos
}

resultados = {}
for versao in PROMPTS:
    regras, exemplos = PROMPTS[versao]
    print("===== Prompt", versao)
    resultados[versao] = avaliar(regras, exemplos, CASOS)
    print()

print("Resumo:")
for versao in resultados:
    print("  ", versao, "->", resultados[versao], "de", len(CASOS))

Guarde esta tabela: ela vai para o relatório do Marco 2 (versão, o que mudou, acertos).

---
# Etapa 6 — Ligando o modelo ao A\*

## O A\* do mapa da Romênia

É uma versão simples do A\* das aulas de busca. **No projeto, vocês usam o A\* do trio**, do Marco 1.

A heurística `h` é a distância em linha reta **até Bucareste** (tabela do livro). Para outros destinos, usamos `h = 0`: o A\* vira busca de custo uniforme, que também encontra o caminho mais curto.

In [ ]:
ESTRADAS = {
    ("Arad", "Zerind"): 75, ("Arad", "Sibiu"): 140, ("Arad", "Timisoara"): 118,
    ("Zerind", "Oradea"): 71, ("Oradea", "Sibiu"): 151, ("Timisoara", "Lugoj"): 111,
    ("Lugoj", "Mehadia"): 70, ("Mehadia", "Drobeta"): 75, ("Drobeta", "Craiova"): 120,
    ("Craiova", "Rimnicu Vilcea"): 146, ("Craiova", "Pitesti"): 138, ("Sibiu", "Fagaras"): 99,
    ("Sibiu", "Rimnicu Vilcea"): 80, ("Rimnicu Vilcea", "Pitesti"): 97, ("Fagaras", "Bucareste"): 211,
    ("Pitesti", "Bucareste"): 101, ("Bucareste", "Giurgiu"): 90, ("Bucareste", "Urziceni"): 85,
    ("Urziceni", "Hirsova"): 98, ("Hirsova", "Eforie"): 86, ("Urziceni", "Vaslui"): 142,
    ("Vaslui", "Iasi"): 92, ("Iasi", "Neamt"): 87,
}
LINHA_RETA_ATE_BUCARESTE = {
    "Arad": 366, "Bucareste": 0, "Craiova": 160, "Drobeta": 242, "Eforie": 161, "Fagaras": 176,
    "Giurgiu": 77, "Hirsova": 151, "Iasi": 226, "Lugoj": 244, "Mehadia": 241, "Neamt": 234,
    "Oradea": 380, "Pitesti": 100, "Rimnicu Vilcea": 193, "Sibiu": 253, "Timisoara": 329,
    "Urziceni": 80, "Vaslui": 199, "Zerind": 374,
}

# lista de vizinhos de cada cidade (as estradas valem nos dois sentidos)
VIZINHOS = {}
for cidade in CIDADES:
    VIZINHOS[cidade] = []
for (a, b), km in ESTRADAS.items():
    VIZINHOS[a].append((b, km))
    VIZINHOS[b].append((a, km))


def h(cidade, destino):
    if destino == "Bucareste":
        return LINHA_RETA_ATE_BUCARESTE[cidade]
    return 0


def a_estrela(origem, destino):
    fronteira = [(h(origem, destino), 0, origem, [origem])]   # (f, g, cidade, caminho)
    visitadas = []
    while len(fronteira) > 0:
        melhor = min(fronteira)            # o nó com menor f = g + h
        fronteira.remove(melhor)
        f, g, cidade, caminho = melhor
        if cidade == destino:
            return caminho, g
        if cidade in visitadas:
            continue
        visitadas.append(cidade)
        for vizinha, km in VIZINHOS[cidade]:
            if vizinha not in visitadas:
                novo_g = g + km
                fronteira.append((novo_g + h(vizinha, destino), novo_g, vizinha, caminho + [vizinha]))
    return None, None


print(a_estrela("Arad", "Bucareste"))
print(a_estrela("Oradea", "Sibiu"))

## 🛠️ TODO 5 — O agente

A função `agente` junta tudo: modelo → JSON → conferência → A\*. Três passos já estão prontos. **Falta o passo 4:** chamar o A\* com a origem e o destino que vieram nos dados.

*Dica:* `caminho, custo = a_estrela(dados["origem"], dados["destino"])`

In [ ]:
def agente(pedido):
    # passo 1: o modelo transforma o pedido em JSON (prompt v3: formato + exemplos)
    resposta = conversar(montar_mensagens(PROMPT_V3, pedido, EXEMPLOS))
    # passo 2: recortar o JSON
    dados = extrair_json(resposta)
    # passo 3: conferir antes de chamar o A*
    if not validar(dados, CIDADES):
        return "Não entendi a origem ou o destino. Use cidades do mapa da Romênia."
    # passo 4: TODO 5 — chame o A* com a origem e o destino dos dados
    caminho, custo = None, None   # apague esta linha e escreva a sua

    if caminho is None:
        return "Não existe caminho entre essas cidades."
    return "Caminho: " + " -> ".join(caminho) + " | custo: " + str(custo) + " km"


print(agente("Quero ir de Arad até Bucareste."))
print(agente("preciso chegar em sibiu, estou em oradea"))
print(agente("Me leva de Arad para Paris."))

In [ ]:
# --- Verificação do TODO 5 (não altere) ---
r = agente("Quero ir de Arad até Bucareste.")
if "418" in r:
    print("✅ TODO 5 correto. O modelo extraiu, o código conferiu e o A* calculou 418 km.")
else:
    print("❌ Ainda não. Para Arad -> Bucareste o A* deveria devolver 418 km. Saiu:", r)

**Repare em quem faz o quê:** o modelo só traduziu o pedido. O número 418 saiu do A\*. Na próxima aula, o modelo também vai **explicar** o resultado e **decidir** quando chamar o A\*.

---
# Desafio Final (Entrega Obrigatória) — O domínio do seu trio

Agora repita a Etapa 5 com o problema **do seu trio**.

## 🛠️ TODO 6 — Estados, prompt e casos de teste do trio

1. `ESTADOS_DO_TRIO`: os estados (lugares, salas, paradas, tarefas…) que o A\* do trio aceita. **Pelo menos 5.**
2. `PROMPT_DO_TRIO`: o prompt de extração, com as quatro peças (papel, tarefa, contexto, formato). Use `", ".join(ESTADOS_DO_TRIO)` para colocar a lista dentro do prompt.
3. `EXEMPLOS_DO_TRIO`: pelo menos **2** exemplos (pedido, resposta JSON).
4. `CASOS_DO_TRIO`: pelo menos **5** casos de teste, com pelo menos **um** difícil.

> Se o A\* do trio usa outros nomes de campo (por exemplo `inicio` e `objetivo`), mantenha `origem` e `destino` aqui: as funções `extrair_json`, `validar` e `avaliar` usam esses nomes.

In [ ]:
# TODO 6: troque os exemplos abaixo pelo domínio do SEU trio
ESTADOS_DO_TRIO = ["estado 1", "estado 2"]

PROMPT_DO_TRIO = ""

EXEMPLOS_DO_TRIO = [
]

CASOS_DO_TRIO = [
]

print(len(ESTADOS_DO_TRIO), "estados |", len(EXEMPLOS_DO_TRIO), "exemplos |", len(CASOS_DO_TRIO), "casos")

In [ ]:
# --- Verificação do TODO 6 (não altere) ---
problemas = []
if len(ESTADOS_DO_TRIO) < 5 or "estado 1" in ESTADOS_DO_TRIO:
    problemas.append("escreva pelo menos 5 estados do domínio do trio")
if len(PROMPT_DO_TRIO) < 50 or "json" not in PROMPT_DO_TRIO.lower():
    problemas.append("o prompt precisa pedir a resposta em JSON")
if len(EXEMPLOS_DO_TRIO) < 2:
    problemas.append("escreva pelo menos 2 exemplos")
if len(CASOS_DO_TRIO) < 5:
    problemas.append("escreva pelo menos 5 casos de teste")
if len(problemas) == 0:
    print("✅ TODO 6 correto. Agora rode a avaliação na próxima célula.")
else:
    for p in problemas:
        print("❌", p)

In [ ]:
print("===== Prompt do trio, sem exemplos")
sem = avaliar(PROMPT_DO_TRIO, [], CASOS_DO_TRIO)
print()
print("===== Prompt do trio, com exemplos")
com = avaliar(PROMPT_DO_TRIO, EXEMPLOS_DO_TRIO, CASOS_DO_TRIO)

### Sua análise — edite esta célula

Responda em 4 a 6 linhas:

1. **Os números:** quantos casos o prompt do trio acertou sem exemplos e com exemplos?
2. **Os erros:** escolha um caso que o modelo errou. O que ele devolveu e por que você acha que errou?
3. **A conferência:** sem a função `validar`, o que aconteceria se o modelo devolvesse um estado que não existe no A\* do trio?

---

_Escreva sua análise aqui, substituindo esta linha._

---

---
## ✅ Checklist antes de entregar

- [ ] Os seis TODOs com ✅ na verificação.
- [ ] O token do Hugging Face está nos **Segredos** (🔑) e **não aparece** em nenhuma célula.
- [ ] A tabela de resultados das versões v1, v2 e v3 foi gerada.
- [ ] O Desafio Final usa os estados **do seu trio**.
- [ ] A célula **Sua análise** foi preenchida.
- [ ] `Ambiente de execução → Reiniciar sessão e executar tudo` roda sem erro.

## Resumo de fixação

| Conceito da aula | Onde apareceu no notebook |
|---|---|
| Prompt e papéis (system, user, assistant) | `montar_mensagens` |
| Saída estruturada | Prompt v2 e `json.loads` |
| Nunca confiar sem conferir | `extrair_json` (TODO 1) e `validar` (TODO 2) |
| Few-shot | `EXEMPLOS` (TODO 3) |
| Prompts versionados e casos de teste | `PROMPTS`, `CASOS` e `avaliar` (TODO 4) |
| O modelo traduz, o A\* calcula | `agente` (TODO 5) |

---
**Para a próxima aula (28/10 — Agente com ferramenta):** traga o A\* do trio funcionando como `a_estrela(origem, destino)`, devolvendo `(caminho, custo)`, e a lista de estados válidos do domínio. Na aula de 28/10 sai a escala do seminário.